In [2]:
import os
from pathlib import Path
import duckdb
from dotenv import load_dotenv
from sklearn.metrics import log_loss

load_dotenv(dotenv_path=Path(os.getcwd()).resolve().parent / ".env")
from typesafe_sdk import Noul, TypeSafeClient
import pandas as pd
import numpy as np

In [17]:
import duckdb
import numpy as np
import pandas as pd

transactions_path = "../data/raw/transactions.csv"
train_path = "../data/raw/train.csv"
user_logs_path = "../data/raw/user_logs.csv"
members_path = "../data/raw/members_v3.csv"
cutoff_ts = pd.Timestamp("2017-01-31")
size = 50
seed = 42

con = duckdb.connect()

# 1. Sample `size` random users reproducibly from train.csv
con.execute(f"""
    CREATE TEMP TABLE sampled_train AS
    SELECT *
    FROM read_csv_auto('{train_path}')
    USING SAMPLE reservoir({size} ROWS) REPEATABLE ({seed});
""")

df_train = con.execute("SELECT * FROM sampled_train").df()
print(f"Train size: {df_train.shape[0]}")

# 2. Filter CSVs (with <= 20170131 cutoff for train.csv) and fix the LEFT JOIN order
df = con.execute(f"""
    WITH filtered_tx AS (
        SELECT
            t.msno, t.transaction_date, t.membership_expire_date,
            t.is_cancel, t.plan_list_price, t.actual_amount_paid, t.payment_plan_days
        FROM read_csv_auto('{transactions_path}') t
        SEMI JOIN sampled_train s ON t.msno = s.msno
        WHERE t.transaction_date <= 20170131
    ),
    filtered_logs AS (
        SELECT
            u.msno, u.date,
            u.num_25, u.num_50, u.num_75, u.num_985, u.num_100, u.num_unq, u.total_secs
        FROM read_csv_auto('{user_logs_path}') u
        SEMI JOIN sampled_train s ON u.msno = s.msno
        WHERE u.date <= 20170131
    ),
    filtered_members AS (
        SELECT m.msno, m.city, m.registered_via, m.registration_init_time
        FROM read_csv_auto('{members_path}') m
        SEMI JOIN sampled_train s ON m.msno = s.msno
    )
    SELECT
        t.msno,
        strptime(CAST(t.transaction_date AS VARCHAR), '%Y%m%d') AS transaction_date,
        strptime(CAST(t.membership_expire_date AS VARCHAR), '%Y%m%d') AS membership_expire_date,
        t.plan_list_price,
        t.actual_amount_paid,
        t.payment_plan_days,
        strptime(CAST(m.registration_init_time AS VARCHAR), '%Y%m%d') AS registration_init_time,
        COALESCE(m.city, -1) AS city,
        COALESCE(m.registered_via, -1) AS registered_via,
        t.is_cancel,
        COALESCE(u.num_25, 0) AS num_25,
        COALESCE(u.num_50, 0) AS num_50,
        COALESCE(u.num_75, 0) AS num_75,
        COALESCE(u.num_985, 0) AS num_985,
        COALESCE(u.num_100, 0) AS num_100,
        COALESCE(u.num_unq, 0) AS num_unq,
        COALESCE(u.total_secs, 0.0) AS total_secs
    FROM filtered_tx t
    LEFT JOIN filtered_logs u
        ON t.msno = u.msno AND t.transaction_date = u.date
    LEFT JOIN filtered_members m
        ON t.msno = m.msno
    ORDER BY t.msno, t.transaction_date
""").df()


def validate_pipeline(df: pd.DataFrame, df_train: pd.DataFrame, cutoff_date: str = "2017-01-31"):
    print("=== 1. Shape & User Coverage ===")
    sampled_users = df_train["msno"].nunique()
    matched_users = df["msno"].nunique()
    print(f"Total rows: {len(df):,}")
    print(f"Users with transactions <= {cutoff_date}: {matched_users} / {sampled_users}")
    if matched_users < sampled_users:
        print(f"  [!] {sampled_users - matched_users} sampled users have NO transactions before cutoff.")

    print("\n=== 2. Missing Values (NaN) & Infinities ===")
    nan_counts = df.isna().sum()
    nan_cols = nan_counts[nan_counts > 0]
    if nan_cols.empty:
        print("OK: No NaN values found.")
    else:
        print("Columns with NaNs:")
        print(nan_cols)

    num_cols = df.select_dtypes(include=[np.number]).columns
    inf_counts = np.isinf(df[num_cols]).sum()
    inf_cols = inf_counts[inf_counts > 0]
    if not inf_cols.empty:
        print("  [!] Infinite values found:")
        print(inf_cols)

    print("\n=== 3. Temporal & Leakage Checks ===")
    cutoff_ts = pd.Timestamp(cutoff_date)
    leaked_rows = (df["transaction_date"] > cutoff_ts).sum()
    print(f"Transaction date range : {df['transaction_date'].min().date()} -> {df['transaction_date'].max().date()}")
    print(
        f"Expire date range      : {df['membership_expire_date'].min().date()} -> {df['membership_expire_date'].max().date()}")
    if df["registration_init_time"].notna().any():
        print(
            f"Registration date range: {df['registration_init_time'].min().date()} -> {df['registration_init_time'].max().date()}")

    assert leaked_rows == 0, f"DATA LEAKAGE: {leaked_rows} rows have transaction_date > {cutoff_date}!"
    print(f"OK: Zero temporal leakage (all transaction_date <= {cutoff_date}).")

    # Sanity-check date ordering anomalies (common in KKBox cancellations/refunds)
    neg_duration = (df["membership_duration"] < 0).sum()
    pre_reg_tx = (df["transaction_date"] < df["registration_init_time"]).sum()
    print(f"Rows with expire_date < transaction_date (cancellations/adjustments): {neg_duration}")
    print(f"Rows with transaction_date < registration_init_time: {pre_reg_tx}")

    # Verify chronological sorting per user
    is_sorted = df.groupby("msno")["transaction_date"].is_monotonic_increasing.all()
    assert is_sorted, "ERROR: Transactions are not sorted chronologically per msno!"
    print("OK: Transactions are strictly sorted per user.")


# 3. Feature engineering (JSON-safe scalars and day integers)
df["discount"] = df["plan_list_price"] - df["actual_amount_paid"]
df["amount_per_day"] = np.where(
    df["payment_plan_days"] > 0,
    df["actual_amount_paid"] / df["payment_plan_days"],
    0.0,
)
df["membership_duration"] = (df["membership_expire_date"] - df["transaction_date"]).dt.days
df["registration_duration"] = (
    (df["membership_expire_date"] - df["registration_init_time"]).dt.days.fillna(-1).astype(int)
)
validate_pipeline(df, df_train, cutoff_date="2017-01-31")
df["days_since_tx"] = (cutoff_ts - df["transaction_date"]).dt.days
df["days_to_expire"] = (df["membership_expire_date"] - cutoff_ts).dt.days
# Convert Timestamps to strings for clean JSON serialization
df["transaction_date"] = df["transaction_date"].dt.strftime("%Y-%m-%d")
df["membership_expire_date"] = df["membership_expire_date"].dt.strftime("%Y-%m-%d")


# 4. Group into JSON list with scalar aggregates
cols = [
    "transaction_date",
    "membership_expire_date",
    "is_cancel",
    "actual_amount_paid",
    "discount",
    "amount_per_day",
    "membership_duration",
    "num_25",
    "num_50",
    "num_75",
    "num_985",
    "num_100",
    "num_unq",
    "total_secs",
]

user_cols = [
    "city",
    "registered_via",
    "registration_duration",
]

def build_aggregated(group: pd.DataFrame) -> dict:
    total_songs = (
        group["num_25"].sum()
        + group["num_50"].sum()
        + group["num_75"].sum()
        + group["num_985"].sum()
        + group["num_100"].sum()
    )

    return {
        "total_transactions": int(len(group)),
        "cancel_count": int(group["is_cancel"].sum()),
        "cancel_rate": float(group["is_cancel"].mean()),
        'total_songs': int(total_songs),
        # Uncomment if you add t.is_auto_renew to your DuckDB query:
        # "auto_renew_rate": float(group["is_auto_renew"].mean()),
        # "last_is_auto_renew": int(group["is_auto_renew"].iloc[-1]),
        "days_since_last_tx": int(group["days_since_tx"].iloc[-1]),
        "days_until_expire": int(group["days_to_expire"].iloc[-1]),
        "avg_membership_duration": float(group["membership_duration"].mean()),
        "total_amount_paid": float(group["actual_amount_paid"].sum()),
        "avg_amount_paid": float(group["actual_amount_paid"].mean()),
        "total_discount": float(group["discount"].sum()),
        "unique_plans": int(group["payment_plan_days"].nunique()),
        "avg_total_sec": float(group["total_secs"].mean()),
        "std_total_sec": float(group["total_secs"].std(ddof=0)),
        "active_tx_days_ratio": float((group["total_secs"] > 0).mean()),
        "avg_unique_songs": float(group["num_unq"].mean()),
        "song_completion_ratio": float(group["num_100"].sum() / total_songs) if total_songs > 0 else 0.0,
        "song_skip_ratio": float(group["num_25"].sum() / total_songs) if total_songs > 0 else 0.0,
    }

json_list = [
    {
        msno: {
            "user_info": {
                "city": int(group["city"].iloc[0]),
                "registered_via": int(group["registered_via"].iloc[0]),
                "registration_duration": int(group["registration_duration"].iloc[-1]),
            },
            # "transactions": group[cols].to_dict(orient="records"),
            "aggregated": build_aggregated(group),
        }
    }
    for msno, group in df.groupby("msno", sort=False)
]

json_list = [
    {
        msno: {
            "user_info": {
                "city": int(group["city"].iloc[0]),
                "registered_via": int(group["registered_via"].iloc[0]),
                "registration_duration": int(group["registration_duration"].iloc[-1]),
            },
            # "transactions": group[cols].to_dict(orient="records"),
            "aggregated": build_aggregated(group),
        }
    }
    for msno, group in df.groupby("msno", sort=False)
]
print(json_list[0])

Train size: 50
=== 1. Shape & User Coverage ===
Total rows: 700
Users with transactions <= 2017-01-31: 50 / 50

=== 2. Missing Values (NaN) & Infinities ===
Columns with NaNs:
registration_init_time    73
dtype: int64

=== 3. Temporal & Leakage Checks ===
Transaction date range : 2015-01-02 -> 2017-01-31
Expire date range      : 2015-02-03 -> 2017-03-24
Registration date range: 2009-02-08 -> 2017-01-11
OK: Zero temporal leakage (all transaction_date <= 2017-01-31).
Rows with expire_date < transaction_date (cancellations/adjustments): 2
Rows with transaction_date < registration_init_time: 0
OK: Transactions are strictly sorted per user.
{'+3lPIF5UkujvJ2nqWJUPUHvkx9U1FsWG6sk0vQDoNqg=': {'user_info': {'city': 1, 'registered_via': 7, 'registration_duration': 1156}, 'aggregated': {'total_transactions': 14, 'cancel_count': 0, 'cancel_rate': 0.0, 'days_since_last_tx': 4, 'days_until_expire': 27, 'avg_membership_duration': 30.5, 'total_amount_paid': 1386.0, 'avg_amount_paid': 99.0, 'total_disc

In [29]:
instructions = """
Your job is to forecast if a user WILL NOT make a new service subscription transaction within 30 days after the current membership expiration date (i.e., whether the user will churn).

You will be given a chronological list of transaction records for a user, where each record is a dictionary containing transaction details and the user's listening activity on that transaction date:
- transaction_date: Date of the transaction (YYYYMMDD)
- membership_expire_date: Expiration date of the membership (YYYYMMDD)
- is_cancel: Whether the user canceled the membership in this transaction (1 = canceled, 0 = not canceled) (subscription cancellation does not imply the user has churned )
- num_25: Number of songs played less than 25% of the song length
- num_50: Number of songs played between 25% and 50% of the song length
- num_75: Number of songs played between 50% and 75% of the song length
- num_985: Number of songs played between 75% and 98.5% of the song length
- num_100: Number of songs played over 98.5% of the song length
- num_unq: Number of unique songs played
- total_secs: Total seconds of music played (0.0 indicates no recorded listening activity on that date)
"""

results = {}

with TypeSafeClient(model="jev-latest") as client:
    for row in json_list:
        msno = list(row.keys())[0]
        transactions = list(row.values())
        print(f"Handling transaction for a user: {msno}")
        state = transactions
        response = client.system_one(
            state=state,
            questions={
                "is_churn": Noul(
                    instructions=instructions,
                ),
            },
        )
        print(f'Response: {response.answers["is_churn"].noul}')
        results[msno] = response.answers["is_churn"].noul

y_true = (
    df_train.set_index("msno")["is_churn"]
    .reindex(results.keys())
    .dropna()
)
y_pred = [results[msno] for msno in y_true.index]
loss = log_loss(y_true.values, y_pred, labels=[0, 1])

print(list(zip(y_true.values, y_pred)))
print(f"Log Loss: {loss:.5f}")

[{'transaction_date': 20160128, 'membership_expire_date': 20160227, 'is_cancel': 0, 'num_25': 124, 'num_50': 321, 'num_75': 57, 'num_985': 14, 'num_100': 6, 'num_unq': 503, 'total_secs': 44605.105}, {'transaction_date': 20160227, 'membership_expire_date': 20160327, 'is_cancel': 0, 'num_25': 1, 'num_50': 3, 'num_75': 0, 'num_985': 1, 'num_100': 5, 'num_unq': 10, 'total_secs': 1726.961}, {'transaction_date': 20160327, 'membership_expire_date': 20160427, 'is_cancel': 0, 'num_25': 0, 'num_50': 0, 'num_75': 0, 'num_985': 0, 'num_100': 0, 'num_unq': 0, 'total_secs': 0.0}, {'transaction_date': 20160427, 'membership_expire_date': 20160527, 'is_cancel': 0, 'num_25': 0, 'num_50': 0, 'num_75': 0, 'num_985': 0, 'num_100': 0, 'num_unq': 0, 'total_secs': 0.0}, {'transaction_date': 20160527, 'membership_expire_date': 20160627, 'is_cancel': 0, 'num_25': 13, 'num_50': 3, 'num_75': 0, 'num_985': 2, 'num_100': 25, 'num_unq': 17, 'total_secs': 7325.04}, {'transaction_date': 20160627, 'membership_expire_da